In [6]:
import torch
import torchvision.models as models
import torchvision.transforms as T
from PIL import Image
import requests
import numpy as np
import matplotlib.pyplot as plt
from io import BytesIO

In [4]:
def download_image(url):
  #download image from the url given
  headers  = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.447'}
  response = requests.get(url, headers=headers)
  return Image.open(BytesIO(response.content)).convert('RGB')

In [3]:
def load_imagenet_labels():
  "Donwload the 1000 imagenet labels"
  labels_path = 'https://raw.githubusercontent.com/anishathalye/imagenet-simple-labels/master/imagenet-simple-labels.json'
  return requests.get(labels_path).json()

In [46]:
def preprocess_image(image,target_size=224):
  transform = T.Compose([
    T.Resize(target_size),
    T.CenterCrop(target_size),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
  ])
  return transform(image).unsqueeze(0)

In [49]:
from torch.autograd.grad_mode import no_grad
from torchvision.models.resnet import resnet50, ResNet50_Weights

def classify_image_using_resnet(image_url, topk = 5):
    pil_img = download_image(image_url)

    img = preprocess_image(pil_img)

    # Use the modern weights parameter to avoid deprecation warnings
    model = resnet50(weights=ResNet50_Weights.DEFAULT)
    model.eval() #set model to evaluation mode

    with no_grad():
      output = model(img)

    labels = load_imagenet_labels()
    probabilities = torch.nn.functional.softmax(output[0], dim=0)
    top_probabilities, top_indices = torch.topk(probabilities, topk)
    print(f"\n Image URL:{image_url}")
    print("Top Predictions:")
    for i in range(topk):
      class_idx = top_indices[i].item()
      prob = top_probabilities[i].item()
      print(f"  {labels[class_idx]:<30} ({prob:.4f})")

In [ ]:
test_image_url =""
classify_image_using_resnet(test_image_url, topk=5)
